# Create city networks from pbf

This notebook loads pbf files of single European countries via pyrosm, then uses city boundary geojson files of all European cities (above 100k population) to extract and export local street and bike networks as gpkg files from the corresponding country.

## Preliminary step

All .osm.pbf country files need to be downloaded from [geofabrik](https://download.geofabrik.de/europe.html), renamed to just `countryname.osm.pbf`, and placed in the countries folder. The following country names need special renames to the following:
- north-macedonia.osm.pbf
- bosnia-and-herzegovina.osm.pbf
- ireland.osm.pbf

## Load packages

In [ ]:
import pyrosm
import osmnx as ox
import networkx as nx
import csv
from growbikenet.functions import *
from growbikenet import constants
from growbikenet import settings
from slugify import slugify

## Parameters

In [ ]:
countries_path = "../countries/"
boundaries_folder = "boundaries/"
cities_path = "../cities/"
cityfilename = "european_capitalsand100000pop.csv"
output_folder = "cityexport/"
dropnodedata = ['osmid','tags','version','changeset','visible'] # osmid is necessary to drop, as will be reinserted
dropedgedata = ['key', 'bicycle','busway','cycleway','est_width','foot','int_ref','lit','motor_vehicle','oneway:bicycle','passing_places','sidewalk','smoothness','surface','tracktype','width','timestamp','version','tags','osm_type']

## Load cities

In [ ]:
with open(cities_path+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {slugify(rows[0]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2], header[3]: rows[3], "boundaryfile": slugify(rows[0])+"_"+slugify(rows[3])} for rows in reader}

## Run

In [ ]:
passuntil = None # Set to a city name to ignore all previous cities. Set None to not ignore any cities. 
for cityid, city_info in cities.items():
    if city_info["name_en"] == passuntil:
        passuntil = None
    if passuntil is None:
        print(city_info["name_en"]+", "+city_info["country_en"])
        # Load city boundary
        try:
            boundary = gpd.read_file(cities_path+boundaries_folder+city_info["boundaryfile"]+".geojson")
        except:
            boundary = gpd.read_file(cities_path+boundaries_folder+city_info["boundaryfile"]+".shp")
    
        # =======================================
        # Create city osm object from country pbf
        countrypbf_file = countries_path+slugify(city_info["country_en"])+".osm.pbf"
        citypbf_file = cities_path+output_folder+"pbfs/"+city_info["boundaryfile"]+".osm.pbf"
        citygpkg_file_street_network = cities_path+output_folder+"street_networks/"+city_info["boundaryfile"]+".gpkg"
        citygpkg_file_bike_network = cities_path+output_folder+"bike_networks/"+city_info["boundaryfile"]+".gpkg"
        if not os.path.exists(citypbf_file): # Crop country osm object to envelope of city boundary, save as pbf, and re-load as city osm object
            osm_country = pyrosm.OSM(countrypbf_file, bounding_box=boundary.loc[0, 'geometry'])
            osm_country.to_pbf(output_path=citypbf_file)
        osm = pyrosm.OSM(citypbf_file)
    
        # =======================================
        # Extract street network from pyrosm's city osm object, truncate to boundary, and save as city gpkg street network
        if not os.path.exists(citygpkg_file_street_network): # Create and save
            (street_nodes, street_edges) = osm.get_network(network_type="driving", nodes=True)
            street_network = pyrosm.OSM.to_graph(street_nodes, street_edges, graph_type='networkx', from_id_col='u', to_id_col='v', edge_id_col='id', node_id_col='id', retain_all=False, osmnx_compatible=True, 
                                                 simplify=False)
            street_network = ox.truncate.truncate_graph_polygon(street_network, boundary.to_crs("4326").loc[0, 'geometry'])
            street_network = ox.simplification.simplify_graph(street_network)
            street_network = ox.truncate.largest_component(street_network)
            street_network.remove_nodes_from(list(nx.isolates(street_network)))
            for u, data in street_network.nodes(data=True):
                for k in dropnodedata:
                    if k in data:
                        del data[k]
            for u, v, data in street_network.edges(data=True):
                for k in dropedgedata:
                    if k in data:
                        del data[k]
            street_network = nx.MultiGraph(ox.convert.to_digraph(street_network))
            ox.io.save_graph_geopackage(street_network, citygpkg_file_street_network)
        
    
        # =======================================
        # Extract bike network from pyrosm's city osm object, truncate to boundary, and save as city gpkg bike network
        if not os.path.exists(citygpkg_file_bike_network): # Create and save
            (bike_nodes, bike_edges) = osm.get_network(network_type="cycling", nodes=True, custom_filter=constants.PBI_CUSTOM_FILTER)
            if bike_nodes is not None and bike_edges is not None:
                bike_network = pyrosm.OSM.to_graph(bike_nodes, bike_edges, graph_type='networkx', from_id_col='u', to_id_col='v', edge_id_col='id', node_id_col='id', force_bidirectional=True, retain_all=True, osmnx_compatible=True,
                                                   simplify=False)
                try: # Case Horlivka
                    bike_network = ox.truncate.truncate_graph_polygon(bike_network, boundary.to_crs("4326").loc[0, 'geometry'])
                except ValueError:
                    print("-- No bike network found inside city boundary")
                    continue
                bike_network = ox.simplification.simplify_graph(bike_network)
                bike_network.remove_nodes_from(list(nx.isolates(bike_network)))
                for u, data in bike_network.nodes(data=True):
                    for k in dropnodedata:
                        if k in data:
                            del data[k]
                for u, v, data in bike_network.edges(data=True):
                    for k in dropedgedata:
                        if k in data:
                            del data[k]
                bike_network = nx.MultiGraph(ox.convert.to_digraph(bike_network))
                ox.io.save_graph_geopackage(bike_network, citygpkg_file_bike_network)
            else:
                pass # pyrosm issues a warning